# 🎯 [K12_M07] 시험 질문으로 챗봇 점검하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m07_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 07. 시험 질문으로 챗봇 점검하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 07. 시험 질문으로 챗봇 점검하기
**소요시간**: 약 1.5시간

![미션 7 12대 벤치마크 평가 질의 정량 채점 기준 정답표 및 프롬프트 재튜닝 프레임워크 다이어그램](/uploads/common/K12_m07_benchmark_eval.png)

### 📋 미션 개요 및 배경
행정 FAQ 챗봇의 신뢰성과 응답 정확도를 객관적으로 검증하기 위해 정량적 벤치마크 테스트 세트를 설계합니다. 정상 절차 질의 8건과 FAQ 외 예외 질의 4건으로 구성된 12개 테스트 세트를 마련하고, 기준 정답과 AI 응답을 비교하여 정량적 정확도를 산출합니다. 오답 유형을 분석해 시스템 프롬프트의 결함을 보완하고 재시험하는 테스트 주도 최적화(Evaluation & Refinement) 역량을 습득합니다.

### 🎯 미션 학습목표
시험 질문 12개(정상 8개, 자료 밖 4개)와 정답을 만들고 챗봇 답을 정답과 비교해 정확도를 계산합니다. 틀린 질문을 유형별로 나누어 프롬프트를 한 번 고친 뒤 다시 시험합니다. **이를 통해 테스트 데이터셋(Benchmark)을 기반으로 정량적 정확도를 측정하고 오답 패턴을 분석하여 프롬프트를 반복 튜닝하는 품질 평가·개선 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 시험 질문 12개(정상 8개, 자료 밖 4개)를 구성한다.
#### 이 단계의 역할

챗봇의 성능과 신뢰성을 객관적으로 측정하기 위해 정상 절차 질의 8개와 자료 밖 예외 질의 4개로 구성된 12개 벤치마크 테스트셋을 설계합니다.

#### 해야 할 일

- 7대 행정 영역(휴가, 증명서, 전입전출, 계정, 시설수리 등)에서 골고루 8개 정상 질문을 선별합니다.

- FAQ에 없는 실제 빈출 착각 질문 4개(수수료, 복무단축, 사적 규정 등)를 예외 질문으로 구성합니다.

- 12개 질문의 ID(T01~T12), 질문 유형, 테스트 목적을 표로 정리합니다.

#### 💡 AI 프롬프트 질의 예시

> **[챗봇 품질 점검용 벤치마크 테스트셋 개발]:** "행정 FAQ 챗봇을 체계적으로 평가하기 위해 [정상 질의 8건(휴가, 증명서, 계정, 시설수리 등 각 영역 골고루 안배), FAQ 범위 밖 예외 질의 4건] 총 12건의 평가 질의 세트를 구성해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **영역 안배:** 특정 행정 분야에 편중되지 않고 전반적인 행정 기능이 망라되었는가?
- **예외 질의 현실성:** 자료 밖 질문이 실제 장병들이 혼동하기 쉬운 행정 주제인가?

#### 핵심 확인 지표

> 시험 질문 12종 벤치마크 목록표

#### 학습자가 제출해야 하는 내용

- 시험 질문 12개 정의서

#### 🔹 Step 2. 시험 질문 12개에 대한 기준 정답표를 작성한다.
#### 이 단계의 역할

각 시험 질문에 대해 FAQ 원문에 엄격히 근거한 모범 기준 정답과 예외 질문에 대한 정상 거절/부서 안내 기준을 사전 정의합니다.

#### 해야 할 일

- 정상 질문 8개에 대해 원문 FAQ의 절차, 서류, 기한, 부서, 근거번호가 포함된 모범 정답을 작성합니다.

- 예외 질문 4개에 대해 'FAQ에 없음' 고지 문구와 올바른 안내 부서명이 명시된 기준을 수립합니다.

- 각 문항별 채점 기준(필수 포함 키워드)을 정답표에 명기합니다.

#### 💡 AI 프롬프트 질의 예시

> **[시험 질문 12종 기준 정답표 작성]:** "구성된 12개 시험 질문에 대해 챗봇이 출력해야 할 '기준 정답(Ground Truth)'을 작성해 줘. 정상 질문은 필수 절차/서류/부서/근거번호를 포함하고, 예외 질문은 거절 문구와 올바른 안내 부서를 명시해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **원문 일치성:** 기준 정답의 날짜, 번호, 절차 순서가 원본 FAQ 표와 100% 일치하는가?
- **채점 기준 명확성:** PASS 판정을 위한 필수 키워드가 구체적으로 정의되었는가?

#### 핵심 확인 지표

> 시험 질문 12종 기준 정답표 (Ground Truth)

#### 학습자가 제출해야 하는 내용

- 기준 정답표 완성본

#### 🔹 Step 3. 챗봇 답과 비교해 정확도를 계산한다.
#### 이 단계의 역할

챗봇에게 12개 질문을 질의하고, 생성된 답변을 기준 정답과 대조하여 PASS/FAIL을 판정하고 종합 정확도(%)를 산출합니다.

#### 해야 할 일

- 12개 시험 질문을 순차적으로 챗봇에 입력하여 실제 답변을 수집합니다.

- 기준 정답표와 챗봇의 답변을 1:1 비교하여 문항별 PASS/FAIL을 판정합니다.

- (PASS 문항 수 / 12) * 100 공식을 통해 정량적 정확도(%) 지표를 계산합니다.

#### 💡 AI 프롬프트 질의 예시

> **[정답 대조 및 정확도 정량 채점 질의]:** "다음 [기준 정답표]와 [챗봇 실제 응답 12건]을 1:1로 비교 채점해 줘. 각 문항별 일치 여부(PASS/FAIL)와 감점 요인을 표로 작성하고, 전체 정답률(정확도 %)을 계산해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **엄격한 채점:** 근거 번호 누락, 서류 1건 누락, 허위 정보 섞임 발생 시 즉시 FAIL 처리
- **정확한 수치 산출:** 백분율 계산 및 감점 사유가 명확히 기록되었는가?

#### 핵심 확인 지표

> 12개 문항 채점 결과표 및 종합 정확도(%)

#### 학습자가 제출해야 하는 내용

- 정량 평가 채점표 완성본

#### 🔹 Step 4. 틀린 질문 유형을 나누고 프롬프트를 고친다.
#### 이 단계의 역할

오답이 발생한 문항을 유형별(서류 누락, 단계 축약, 예외 거절 실패 등)로 분류하고 원인을 진단하여 시스템 프롬프트를 보강한 뒤 재시험합니다.

#### 해야 할 일

- FAIL 판정을 받은 문항의 오류 유형을 분류합니다.

- 오류를 유발한 프롬프트의 취약한 지시문을 식별하고 명확한 보강 규칙을 작성합니다.

- 개선된 프롬프트로 틀린 문항을 재시험하여 PASS로 개선되었는지 확인합니다.

#### 💡 AI 프롬프트 질의 예시

> **[오답 유형 분석 및 프롬프트 튜닝 질의]:** "테스트에서 실패한 문항들의 오류 패턴을 분석해 줘. 동일한 실수를 방지하기 위해 시스템 프롬프트의 지시문과 제약조건을 어떻게 보강해야 하는지 개선 문안을 제안해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **재시험 개선율:** 프롬프트 수정 후 재시험에서 기존 오답 문항이 정상 응답으로 전환되었는가?
- **회귀 테스트:** 수정된 프롬프트가 기존에 잘 맞히던 다른 정상 질문에 악영향을 주지 않았는가?

#### 핵심 확인 지표

> 오류 분석 및 프롬프트 튜닝 전후 개선 보고서

#### 학습자가 제출해야 하는 내용

- 프롬프트 튜닝 내역 및 재시험 결과서

#### 🔹 Step 5. 심화 도전 과제 및 최적화
#### 이 단계의 역할

질문에 오타가 극단적으로 심하거나 여러 질문을 한 번에 물어보는 복합 질의 2건을 추가 테스트하여 견고성을 확인합니다.

#### 해야 할 일

- 고난도 스트레스 질문 2건(예: 오타 범벅 질의, 이중 행정 절차 문의)을 출제합니다.

- 개선된 프롬프트가 고난도 환경에서도 안정적으로 통제되는지 검증합니다.

#### 💡 AI 프롬프트 질의 예시

> **[고난도 스트레스 테스트 질의]:** "고의적 오타 및 다중 문의가 결합된 극한의 장병 질문 2건을 챗봇에 입력하고, 프롬프트가 무너지지 않고 정상 처리하는지 평가해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **견고성(Robustness):** 난해한 입력에도 환각 없이 원칙을 지키는가?

#### 핵심 확인 지표

> 고난도 스트레스 테스트 메모

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **12대 벤치마크 평가 질의 세트 양식 (정상 8건 : 예외 4건):** 7대 행정 영역을 골고루 대표하는 종합 평가 질의 문항 시트
- **모범 정답표(Ground Truth) 및 채점 루브릭 (Scoring Rubric):** 정답, 부분정답, 오답 채점 기준 및 핵심 키워드(기한, 서류, 부서, 근거) 체크리스트
- **[학습 가이드] LLM 정량 평가(Evaluation) 방법론 및 반복 프롬프트 튜닝:** 정답률(Accuracy %) 산출과 오답 패턴 분류를 통한 시스템 프롬프트 최적화 사이클

**실습 팁:** 정상 질문 8개는 휴가, 증명서, 계정, 시설수리 등 다양한 영역에서 출제하고, 예외 질문 4개는 장병들이 자주 혼동하는 비공식 규정 질문으로 구성하여 엄격히 채점하세요.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 시험·개선 — 12개 벤치마크 테스트셋과 기준 정답(Ground Truth)으로 정확도를 정량 산출하고, 오답 분석을 통해 프롬프트를 튜닝·개선했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 12대 벤치마크 테스트셋 및 모범 정답표(Ground Truth) 완성

- 12개 문항 1:1 정량 채점표 및 초기 정확도 산출

- 취약점 분석 기반 프롬프트 튜닝 및 최종 정확도 90% 이상 달성

#### 채점자 판정 메모

> 1차 시험에서 FAIL이 발생했던 문항이 프롬프트의 어느 지시문 보강을 통해 PASS로 전환되었는지 튜닝 전후 대조표를 확인한다.



In [ ]:
# [미션 07.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_07_prompt = """
[12대 벤치마크 테스트셋 구축 및 정량 채점 질의]
가상 한빛대대 행정 FAQ 챗봇의 성능과 신뢰도를 객관적으로 검증하기 위한 12대 벤치마크 테스트 세트를 구성해 줘.
- 구성: 정상 절차 질의 8문항 + FAQ 외 예외 질의 4문항
- 각 문항별 평가 항목: 사실 정확도, 절차 누락 여부, 6단 서식 준수, 근거 표기, 부서 연계
- 12개 문항의 기준 정답표(Ground Truth)를 작성하고, 모의 테스트 결과 채점표(점수 및 종합 정확도)를 도출해 줘.
"""

mission_07_result = """
## 1. 12대 벤치마크 평가 질의 및 기준 정답표 (Ground Truth)

| 번호 | 유형 | 질문 내용 | 기준 정답 핵심 요건 (Ground Truth) |
| :---: | :---: | :--- | :--- |
| **T01** | 정상 | 정기휴가 신청 기한 및 서류 | 출타 5일 전(D-5), 휴가신청서 1부, 중대 행정반, [근거: FAQ-F01] |
| **T02** | 정상 | 포상휴가 사용 기한 및 부서 | 수여일로부터 6개월 이내, 출타 3일 전, 대대 인사과, [근거: FAQ-F02] |
| **T03** | 정상 | 군 복무확인서 발급 소요 시간 | 신청일 기준 24시간 이내 발급, 대대 인사과, [근거: FAQ-F04] |
| **T04** | 정상 | 운전경력확인서 주행거리 조건 | 배차 실적 누적 1,000km 이상만 경력 산입, 수송부, [근거: FAQ-F05] |
| **T05** | 정상 | 전입 첫날 제출 서류 | 가족관계증명서 1부, 급여통장 사본 1부, 중대 행정반, [근거: FAQ-F07] |
| **T06** | 정상 | 자격증 응시료 지원 한도 | 연간 최대 12만 원 한도, 익월 10일까지 신청, 정훈공보실, [근거: FAQ-F10] |
| **T07** | 정상 | 비밀번호 초기화 대리 신청 여부 | 대리인 신청 불가 (본인 직접 신분증 지참 통신실 방문), [근거: FAQ-F12] |
| **T08** | 정상 | 생활관 형광등 고장 접수처 | 행정반 수리접수대장 작성, 매일 16시 취합 익일 수리, 군수과, [근거: FAQ-F14] |
| **T09** | 예외 | 복무기간 총 연가 일수 | FAQ 미수록 고지 + 인사과(내선 102) 안내 + 추측 금지 |
| **T10** | 예외 | 증명서 발급 수수료 및 계좌 | FAQ 미수록 고지 + 인사과(내선 102) 안내 + 추측 금지 |
| **T11** | 예외 | 복무단축 일수 계산 | FAQ 미수록 고지 + 인사과(내선 102) 안내 + 추측 금지 |
| **T12** | 예외 | 영외 민간 병원 외출 기준 | FAQ 미수록 고지 + 의무실(내선 107)/행정반 안내 + 추측 금지 |

---

## 2. 12문항 채점 결과표 및 종합 정확도

- **채점 기준**: 필수 사실 정보 일치, 근거 표기 여부, Fallback 준수 여부 (각 1점, 오답/누락 시 0점)
- **평가 결과**: 12문항 중 11문항 PASS, 1문항 FAIL (초기 정확도: 91.7%)
  - *오답 사례 (T07)*: '대리 불가' 원칙은 안내했으나 근거 번호 누락으로 FAIL
  - *개선 조치*: 프롬프트에 "부정적 답변(불가 안내) 시에도 반드시 근거 번호 명시" 지시 보강 후 재시험 → 12문항 전원 PASS (정확도 100%)
"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 07. 시험 질문으로 챗봇 점검하기", mission_07_prompt, mission_07_result)

